# Time Series Prediction (TSP) Algorithm — Complete 17-Step Implementation

Full implementation of the TSP algorithm from:
> Lin et al., *Time Series Prediction Algorithm for Intelligent Predictive Maintenance*, IEEE RA-L, 2019

Covers all 17 steps of Fig. 4, including the previously missing:
- **Step 10**: White-noise expansion loop (add samples until autocorrelation is detected)
- **Step 11**: q grid starts at 1 (ARMA combinations, not pure AR)
- **Steps 14–15**: Coefficient significance pruning (1.96 × SE test) + re-estimation
- **Step 16**: Residual Ljung-Box validation

Pre-Alarm Module (PreAM) and Death Correlation Index (DCI) are excluded from this notebook (implemented separately).

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

In [ ]:
# ── Cell 2: Global parameters ─────────────────────────────────────────────────
np.random.seed(42)

SICK_SPEC        = 7.5    # DHI threshold at which sick state begins
DEAD_SPEC        = 10.5   # DHI threshold for dead state
M_MODEL          = 30     # baseline model window (paper: ≥30 satisfies CLT)
FORECAST_HORIZON = 250    # steps ahead to forecast RUL
MAX_WN_EXTRA     = 30     # Step 10: max extra samples to add when white noise
CONF_LEVEL       = 1.96   # 95% confidence interval z-score (Steps 14, 16)

In [ ]:
# ── Cell 3: Synthetic throttle-valve data (mirrors paper Example I) ───────────
# Four phases: healthy growth → flat/sick → sudden rise → oscillation near dead
pre_n   = 30
total_n = pre_n + 109
spans   = np.arange(267, 267 + total_n)

YT = np.zeros(total_n)

# Phase 1 – pre-sick (healthy, slight upward drift)
for i in range(pre_n):
    YT[i] = 5.0 + 0.008 * i + np.random.normal(0, 0.06)

# Phase 2 – flat sick behaviour (slow rise)
flat_end = pre_n + 82
for i in range(pre_n, flat_end):
    YT[i] = YT[pre_n - 1] + 0.012 * (i - pre_n) + np.random.normal(0, 0.10)

# Phase 3 – sudden rise toward dead spec
rise_n = 6
for i in range(flat_end, flat_end + rise_n):
    YT[i] = YT[flat_end - 1] + (i - flat_end + 1) * 1.90 + np.random.normal(0, 0.30)

# Phase 4 – oscillation around / above dead spec
osc_start = flat_end + rise_n
for i in range(osc_start, total_n):
    rel     = i - osc_start
    YT[i]   = 12.8 + 2.6 * np.sin(0.9 * rel) + 0.08 * rel + np.random.normal(0, 0.5)
    YT[i]   = max(YT[i], DEAD_SPEC - 0.4)

# Sick state starts at index pre_n (first time DHI would cross 0.7 threshold)
SICK_IDX = pre_n
print(f"Data ready: {total_n} spans. Sick onset at index {SICK_IDX} (span {spans[SICK_IDX]}).")

In [ ]:
# ── Cell 4: TSP helper functions ──────────────────────────────────────────────

# ── Step 10 ──────────────────────────────────────────────────────────────────
def expand_until_not_wn(YM_base, YT_full, sick_idx, max_extra=MAX_WN_EXTRA):
    """
    Step 10 (Lin et al. Fig. 4): if YM is white noise, keep appending
    post-sick samples until autocorrelation is detected or budget runs out.

    Returns
    -------
    YM_out   : expanded sample array
    expanded : True if expansion was actually performed
    """
    YM_out = YM_base.copy()
    lb_lag = max(1, min(10, len(YM_out) // 4))
    lb_p   = acorr_ljungbox(YM_out, lags=[lb_lag],
                            return_df=True)["lb_pvalue"].values[0]
    if lb_p <= 0.05:
        # Already correlated — no expansion needed
        return YM_out, False

    print(f"    [Step 10] YM is white noise (p={lb_p:.4f}). "
          f"Expanding up to +{max_extra} samples ...")

    for extra in range(1, max_extra + 1):
        idx = sick_idx + extra
        if idx >= len(YT_full):
            break
        YM_out = YT_full[: idx + 1].copy()
        lb_lag = max(1, min(10, len(YM_out) // 4))
        lb_p   = acorr_ljungbox(YM_out, lags=[lb_lag],
                                return_df=True)["lb_pvalue"].values[0]
        if lb_p <= 0.05:
            print(f"    [Step 10] Autocorrelation found after +{extra} samples "
                  f"(p={lb_p:.4f}). New M={len(YM_out)}.")
            return YM_out, True

    print(f"    [Step 10] Could not remove white-noise in budget. "
          f"Using M={len(YM_out)}.")
    return YM_out, True


# ── Steps 14 + 15 ────────────────────────────────────────────────────────────
def prune_and_refit(YM_w, order, conf=CONF_LEVEL, verbose=True):
    """
    Steps 14–15 (Lin et al. §IV): drop ARIMA coefficients whose absolute
    value falls within the 95 % confidence interval (|coef| ≤ 1.96·SE),
    then re-estimate the pruned model.

    Parameters
    ----------
    YM_w  : array — (possibly log-transformed) modelling window
    order : (p, d, q) tuple — BIC-selected order from Step 13
    conf  : z-score threshold (default 1.96 for 95 %)

    Returns
    -------
    final_res   : fitted ARIMAResults object
    final_order : (p, d, q) tuple after pruning
    """
    p, d, q = order
    res     = ARIMA(YM_w, order=order).fit(
                  method_kwargs={"warn_convergence": False})
    names   = list(res.param_names)
    params  = res.params
    bse     = res.bse

    keep_ar, keep_ma, dropped = [], [], []
    for i, name in enumerate(names):
        if name.startswith("ar.L"):
            lag = int(name.split("L")[1])
            if abs(params[i]) > conf * bse[i]:   # Eq. (17) in paper
                keep_ar.append(lag)
            else:
                dropped.append(name)
        elif name.startswith("ma.L"):
            lag = int(name.split("L")[1])
            if abs(params[i]) > conf * bse[i]:   # Eq. (18) in paper
                keep_ma.append(lag)
            else:
                dropped.append(name)

    if verbose:
        print(f"    [Step 14] Significant AR lags: {keep_ar}  "
              f"MA lags: {keep_ma}")
        if dropped:
            print(f"    [Step 14] Insignificant predictors dropped: {dropped}")

    # Nothing to prune
    if len(keep_ar) == p and len(keep_ma) == q:
        if verbose:
            print("    [Step 15] All coefficients significant — model unchanged")
        return res, order

    # Build pruned order and re-estimate (Step 15)
    new_p     = max(keep_ar) if keep_ar else 1
    new_q     = max(keep_ma) if keep_ma else 0
    new_order = (new_p, d, new_q)
    if verbose:
        print(f"    [Step 15] Re-estimating ARIMA{new_order} ...")
    try:
        res2 = ARIMA(YM_w, order=new_order).fit(
                   method_kwargs={"warn_convergence": False})
        return res2, new_order
    except Exception:
        if verbose:
            print("    [Step 15] Re-fit failed — keeping original model")
        return res, order


# ── Step 16 ──────────────────────────────────────────────────────────────────
def residual_wn_check(result, m, verbose=True):
    """
    Step 16: Ljung-Box test on model residuals.
    Good fit → residuals should be white noise (p > 0.05).
    """
    resid  = result.resid
    lb_lag = max(1, min(10, m // 4))
    lb_p   = acorr_ljungbox(resid, lags=[lb_lag],
                            return_df=True)["lb_pvalue"].values[0]
    ok     = lb_p > 0.05
    if verbose:
        status = "PASS — residuals are white noise" if ok \
                 else "WARN — residuals still correlated; model may be mis-specified"
        print(f"    [Step 16] Residual Ljung-Box p={lb_p:.4f} → {status}")
    return ok, lb_p


print("Helper functions defined (Steps 10, 14-15, 16).")

In [ ]:
# ── Cell 5: TSP Model Creation — Steps 1–17 ──────────────────────────────────
print("=" * 64)
print("TSP MODEL CREATION  (Lin et al. 2019, Fig. 4)")
print("=" * 64)

# ── Step 1 ───────────────────────────────────────────────────────────────────
# Input YT aging features into DHI module and RUL predictive module
print(f"[Step 1]  YT loaded — {total_n} spans, range "
      f"{spans[0]}–{spans[-1]}")

# ── Step 2 ───────────────────────────────────────────────────────────────────
# DHI < 0.7 triggers TSP model creation (SICK_IDX is where DHI crosses 0.7)
print(f"[Step 2]  DHI crossed 0.7 at index {SICK_IDX} "
      f"(span {spans[SICK_IDX]}) → starting TSP model creation")

# ── Step 3 ───────────────────────────────────────────────────────────────────
# Build modelling window YM from the M samples immediately before sick onset
YM = YT[:SICK_IDX].copy()
print(f"[Step 3]  YM built: M={len(YM)} pre-sick samples "
      f"(spans {spans[0]}–{spans[SICK_IDX-1]})")

# ── Step 4 + 5 ───────────────────────────────────────────────────────────────
# Variance-growth test: if latter half of YM has >2× the variance of the
# first half AND all values are positive → apply log transform
h       = len(YM) // 2
use_log = (np.var(YM[h:]) > 2.0 * np.var(YM[:h])) and np.all(YM > 0)
YM_w    = np.log(YM) if use_log else YM.copy()
print(f"[Step 4]  Variance growth detected: {use_log}")
print(f"[Step 5]  Log transform applied: {use_log}")

# ── Step 6 ───────────────────────────────────────────────────────────────────
# ADF test: H0 = unit root (non-stationary). Reject H0 (p < 0.05) → stationary
adf_stat, adf_p = adfuller(YM_w)[:2]
is_stat = adf_p < 0.05
print(f"[Step 6]  ADF test: stat={adf_stat:.4f}, p={adf_p:.4f} "
      f"→ stationary: {is_stat}")

# ── Step 7 ───────────────────────────────────────────────────────────────────
# If non-stationary, set d=1 (ARIMA handles differencing internally)
d_order = 0
if not is_stat:
    d_order = 1
    print("[Step 7]  Non-stationary → d=1 (1st-order differencing via ARIMA)")
else:
    print("[Step 7]  Already stationary → d=0")

# ── Step 8 ───────────────────────────────────────────────────────────────────
# ACF / PACF: A = lag of max ACF value, B = lag of max PACF value
# These bound the q and p search ranges respectively
max_lag = min(11, len(YM_w) // 3)
acf_v   = np.abs(acf(YM_w,  nlags=max_lag, fft=False)[1:])
pacf_v  = np.abs(pacf(YM_w, nlags=max_lag, method="ols")[1:])
A = min(int(np.argmax(acf_v))  + 1, 10)   # Eq. (12): A = argmax(ρ_k)
B = min(int(np.argmax(pacf_v)) + 1,  9)   # Eq. (13): B = argmax(ρρ_k)
print(f"[Step 8]  ACF max-lag A={A} (q bound), PACF max-lag B={B} (p bound)")

# ── Step 9 ───────────────────────────────────────────────────────────────────
# Ljung-Box test on YM: H0 = white noise (all autocorrelations = 0)
lb_lag = max(1, min(10, len(YM_w) // 4))
lb_p   = acorr_ljungbox(YM_w, lags=[lb_lag],
                        return_df=True)["lb_pvalue"].values[0]
is_wn  = lb_p > 0.05
print(f"[Step 9]  Ljung-Box on YM: p={lb_p:.4f} → white noise: {is_wn}")

# ── Step 10 ──────────────────────────────────────────────────────────────────
# If white noise: expand YM by adding post-sick samples until structure is found.
# After expansion, re-run ACF/PACF to update A and B.
if is_wn:
    YM_w, expanded = expand_until_not_wn(YM_w, YT, SICK_IDX,
                                         max_extra=MAX_WN_EXTRA)
    if expanded:
        max_lag = min(11, len(YM_w) // 3)
        acf_v   = np.abs(acf(YM_w,  nlags=max_lag, fft=False)[1:])
        pacf_v  = np.abs(pacf(YM_w, nlags=max_lag, method="ols")[1:])
        A = min(int(np.argmax(acf_v))  + 1, 10)
        B = min(int(np.argmax(pacf_v)) + 1,  9)
        print(f"    [Step 10] Updated bounds: A={A}, B={B}")
    else:
        # Still white noise after expansion budget exhausted — use minimal order
        A, B = 1, 1
        print("    [Step 10] Fallback: A=B=1")
else:
    print("[Step 10] YM not white noise — no expansion needed")

# ── Steps 11–13 ──────────────────────────────────────────────────────────────
# Create all ARIMA(p, d, q) combinations with p=1..B, q=1..A (paper §IV)
# Also test pure AR (q=0) as an additional candidate.
# Select the combination with the lowest BIC (Eq. 16).
print(f"[Step 11] Building ARIMA combinations: p=1..{B}, q=1..{A} "
      f"+ pure-AR q=0")
print(f"[Step 12] Computing BIC for each combination ...")

best_bic, best_order, best_res = np.inf, (1, d_order, 1), None

# ARMA combinations (q ≥ 1)
for p_try in range(1, min(B + 1, 6)):
    for q_try in range(1, min(A + 1, 5)):
        try:
            r = ARIMA(YM_w, order=(p_try, d_order, q_try)).fit(
                    method_kwargs={"warn_convergence": False})
            if r.bic < best_bic:
                best_bic, best_order, best_res = r.bic, (p_try, d_order, q_try), r
        except Exception:
            pass

# Pure AR candidates (q = 0)
for p_try in range(1, min(B + 1, 6)):
    try:
        r = ARIMA(YM_w, order=(p_try, d_order, 0)).fit(
                method_kwargs={"warn_convergence": False})
        if r.bic < best_bic:
            best_bic, best_order, best_res = r.bic, (p_try, d_order, 0), r
    except Exception:
        pass

print(f"[Step 13] Lowest BIC: ARIMA{best_order}  BIC={best_bic:.3f}")

# ── Steps 14–15 ──────────────────────────────────────────────────────────────
# Prune coefficients not significant at 95 % level, then re-estimate.
print("[Step 14] Checking coefficient significance (|coef| > 1.96 × SE) ...")
final_res, final_order = prune_and_refit(YM_w, best_order, verbose=True)
print(f"[Step 15] Model after pruning: ARIMA{final_order}")

# ── Step 16 ──────────────────────────────────────────────────────────────────
# Validate that residuals of the fitted model are white noise
residual_wn_check(final_res, len(YM_w), verbose=True)

# ── Step 17 ──────────────────────────────────────────────────────────────────
# Confirm the model — store order and transform flag for rolling forecast
TSP_ORDER = final_order
USE_LOG   = use_log

print()
print(f"[Step 17] Model confirmed: ARIMA{TSP_ORDER}  |  log-transform: {USE_LOG}")
print("=" * 64)

In [ ]:
# ── Cell 6: Exponential baseline + rolling TSP forecast ───────────────────────

def fit_exp(t_arr, y_arr):
    """Fit F·exp(C·t) to (t, y) pairs — exponential baseline model."""
    def fn(t, F, C):
        return F * np.exp(C * t)
    try:
        t_n = t_arr - t_arr[0]
        p, _ = curve_fit(fn, t_n, y_arr, p0=[y_arr[0], 0.001],
                         maxfev=6000, bounds=([0, -1], [1e6, 1]))
        return p
    except Exception:
        return np.array([y_arr[0], 0.001])


TSP_YT = np.full(total_n, np.nan)   # rolling 1-step-ahead TSP forecast
EXP_YT = np.full(total_n, np.nan)   # rolling exponential model forecast

print(f"Running rolling forecast from span {spans[SICK_IDX]} "
      f"to {spans[-1]} ({total_n - SICK_IDX} steps) ...")

for t in range(SICK_IDX, total_n):
    avail = YT[: t + 1]

    # ── TSP forecast ─────────────────────────────────────────────────────────
    # Use the TSP-selected ARIMA order; refit on growing history window.
    # Predict FORECAST_HORIZON steps ahead and take the first step as
    # the 1-step estimate for this span (consistent with paper Fig. 6a).
    try:
        seq = np.log(avail) if USE_LOG else avail.copy()
        seq = seq[-60:] if len(seq) > 60 else seq   # cap window to avoid slowdown
        r   = ARIMA(seq, order=TSP_ORDER).fit(
                  method_kwargs={"warn_convergence": False})
        fc          = r.forecast(steps=FORECAST_HORIZON)
        TSP_YT[t]   = np.exp(fc[0]) if USE_LOG else fc[0]
    except Exception:
        TSP_YT[t] = np.nan

    # ── Exponential baseline forecast ────────────────────────────────────────
    # Fit F·exp(C·k) on all sick-state data seen so far; predict next span.
    n_sick = t - SICK_IDX + 1
    if n_sick >= 5:
        exp_F, exp_C = fit_exp(spans[SICK_IDX: t + 1].astype(float),
                               YT[SICK_IDX: t + 1])
    else:
        exp_F, exp_C = YT[SICK_IDX], 0.001

    t_rel      = float(spans[t] - spans[SICK_IDX])
    EXP_YT[t]  = exp_F * np.exp(exp_C * (t_rel + 1))

valid = np.sum(~np.isnan(TSP_YT[SICK_IDX:]))
print(f"Done. TSP valid forecast points: {valid}/{total_n - SICK_IDX}")

In [ ]:
# ── Cell 7: Final comparison plot (replicates paper Fig. 6a style) ────────────
VIS = spans >= 300    # trim the first few pre-sick spans for visual clarity

fig, ax = plt.subplots(figsize=(15, 4.5))
ax.set_facecolor("#fdfdfd")

ax.plot(spans[VIS], YT[VIS],     "r*",  ms=4.5, label="Actual",               zorder=6)
ax.plot(spans[VIS], TSP_YT[VIS], color="deeppink", lw=1.8,
        label=f"TSP-estimated  ARIMA{TSP_ORDER}",  zorder=5)
ax.plot(spans[VIS], EXP_YT[VIS], color="navy", lw=1.8, ls="--",
        label="Exponential-estimated",              zorder=4)

ax.axhline(SICK_SPEC, color="gold",       lw=1.5, ls="--",
           label=f"Sick Spec ({SICK_SPEC})")
ax.axhline(DEAD_SPEC, color="darkorange", lw=2.0,
           label=f"Dead Spec ({DEAD_SPEC})")

ax.set_ylabel(r"$Y_T$")
ax.set_xlabel("Time / span")
ax.set_ylim(0, 22)
ax.grid(alpha=0.25)
ax.legend(loc="upper left", fontsize=8, ncol=3, framealpha=0.85)
ax.set_title("TSP Aging-Feature Prediction — Complete 17-Step Algorithm")
plt.tight_layout()
plt.show()